In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import os
import gc

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
train_path = "/content/drive/MyDrive/credit_scoring_alpha/train_data.parquet"
test_path = "/content/drive/MyDrive/credit_scoring_alpha/test_data.parquet"
train_target_path = "/content/drive/MyDrive/credit_scoring_alpha/train_target.csv"

Я посмотрела список колонок и группирую их

In [ ]:
agg_exprs = [
    pl.col("rn").max().alias("total_loans_count"),

    pl.col(["pre_loans_credit_limit", "pre_loans_outstanding", "pre_loans_max_overdue_sum"]).max().name.suffix("_max"),
    pl.col(["pre_loans_credit_limit", "pre_loans_outstanding", "pre_loans_next_pay_summ"]).mean().name.suffix("_mean"),
    pl.col("pre_loans_total_overdue").sum().alias("total_overdue_sum"),

    pl.col(["pre_since_opened", "pre_pterm", "pre_fterm"]).mean().name.suffix("_mean"),
    pl.col(["pre_loans5", "pre_loans530", "pre_loans3060", "pre_loans6090", "pre_loans90"]).sum().name.suffix("_sum"),
    pl.col(["is_zero_loans5", "is_zero_loans90", "pclose_flag", "fclose_flag"]).mean().name.suffix("_ratio"),
    pl.col(["enc_loans_credit_type", "enc_loans_credit_status", "enc_loans_account_cur"]).n_unique().name.suffix("_nunique"),
]
paym_cols = [f"enc_paym_{i}" for i in range(25)]
agg_exprs.append(pl.col(paym_cols).mean().name.suffix("_mean"))

In [ ]:
# только базовые кредитные фичи, ПОЛНОСТЬЮ исключив enc_paym_0...24 (они топят ОЗУ)
base_cols = [
    "id", "rn", "pre_since_opened", "pre_since_confirmed", "pre_pterm", "pre_fterm",
    "pre_till_pclose", "pre_till_fclose", "pre_loans_credit_limit", "pre_loans_next_pay_summ",
    "pre_loans_outstanding", "pre_loans_total_overdue", "pre_loans_max_overdue_sum",
    "pre_loans_credit_cost_rate", "pre_loans5", "pre_loans530", "pre_loans3060",
    "pre_loans6090", "pre_loans90", "is_zero_loans5", "is_zero_loans530", "is_zero_loans3060",
    "is_zero_loans6090", "is_zero_loans90", "pre_util", "pre_over2limit", "pre_maxover2limit",
    "is_zero_util", "is_zero_over2limit", "is_zero_maxover2limit", "enc_loans_account_holder_type",
    "enc_loans_credit_status", "enc_loans_credit_type", "enc_loans_account_cur", "pclose_flag", "fclose_flag"
]

In [ ]:
# генератор для чтения тяжелого файла мелкими кусками
parquet_file = pq.ParquetFile(train_path)

# промежуточные результаты
chunk_output = "chunk_results.parquet"
if os.path.exists(chunk_output):
    os.remove(chunk_output)

# обрабатываем группы строк (по 200000 строк)
for i in range(parquet_file.num_row_groups):
    chunk = parquet_file.read_row_group(i, columns=base_cols).to_pandas()

    # понижаем точность типов для экономии ОЗУ
    for col in chunk.select_dtypes(include=['float']).columns:
        chunk[col] = chunk[col].astype('float32')
    for col in chunk.select_dtypes(include=['int']).columns:
        chunk[col] = chunk[col].astype('int32')

    # Быстрая базовая агрегация текущего куска данных
    agg_chunk = chunk.groupby("id").agg({
        "rn": "max",
        "pre_loans_credit_limit": "max",
        "pre_loans_outstanding": "mean",
        "pre_loans_total_overdue": "sum",
        "pre_loans90": "sum",
        "is_zero_loans90": "mean",
        "pclose_flag": "mean",
        "fclose_flag": "mean"
    }).reset_index()

    agg_chunk.columns = ['id', 'total_loans_count', 'loans_limit_max', 'loans_outstanding_mean',
                         'total_overdue_sum', 'loans90_sum', 'zero_loans90_ratio', 'pclose_ratio', 'fclose_ratio']

    # Дописываем сжатый чанк в итоговый parquet-файл на диске
    if not os.path.exists(chunk_output):
        agg_chunk.to_parquet(chunk_output, index=False, engine='pyarrow')
    else:
        # Читаем старый кусок, объединяем и перезаписываем (так как id могут дублироваться между группами)
        existing = pd.read_parquet(chunk_output)
        combined = pd.concat([existing, agg_chunk], ignore_index=True)
        # Повторно группируем, чтобы схлопнуть одинаковые ID из разных чанков
        final_combined = combined.groupby("id").agg({
            "total_loans_count": "max", "loans_limit_max": "max", "loans_outstanding_mean": "mean",
            "total_overdue_sum": "sum", "loans90_sum": "sum", "zero_loans90_ratio": "mean",
            "pclose_ratio": "mean", "fclose_ratio": "mean"
        }).reset_index()
        final_combined.to_parquet(chunk_output, index=False, engine='pyarrow')

    del chunk, agg_chunk
    gc.collect()

Всего строк в исходном файле: 18317016
Финальная агрегированная таблица успешно собрана на диске!


In [ ]:
X, y = pd.read_parquet(chunk_output), pd.read_csv(train_target_path)['flag'].squeeze()

Размер итогового датасета для обучения: (2100000, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2100000 entries, 0 to 2099999
Data columns (total 10 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   id                      int32  
 1   total_loans_count       int32  
 2   loans_limit_max         int32  
 3   loans_outstanding_mean  float64
 4   total_overdue_sum       int32  
 5   loans90_sum             int32  
 6   zero_loans90_ratio      float64
 7   pclose_ratio            float64
 8   fclose_ratio            float64
 9   flag                    int64  
dtypes: float64(4), int32(5), int64(1)
memory usage: 120.2 MB


In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(auto_class_weights='Balanced', iterations=100, learning_rate=0.1, verbose=False)

scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    score = roc_auc_score(y_test, preds)
    scores.append(score)

print(f"Средняя ROC-AUC Score: {np.mean(scores):.4f}")


Средняя ROC-AUC Score: 0.5943


In [19]:
parquet_file = pq.ParquetFile(test_path)

chunk_output = "chunk_results.parquet"
if os.path.exists(chunk_output):
    os.remove(chunk_output)

for i in range(parquet_file.num_row_groups):
    chunk = parquet_file.read_row_group(i, columns=base_cols).to_pandas()

    for col in chunk.select_dtypes(include=['float']).columns:
        chunk[col] = chunk[col].astype('float32')
    for col in chunk.select_dtypes(include=['int']).columns:
        chunk[col] = chunk[col].astype('int32')

    agg_chunk = chunk.groupby("id").agg({
        "rn": "max",
        "pre_loans_credit_limit": "max",
        "pre_loans_outstanding": "mean",
        "pre_loans_total_overdue": "sum",
        "pre_loans90": "sum",
        "is_zero_loans90": "mean",
        "pclose_flag": "mean",
        "fclose_flag": "mean"
    }).reset_index()

    agg_chunk.columns = ['id', 'total_loans_count', 'loans_limit_max', 'loans_outstanding_mean',
                         'total_overdue_sum', 'loans90_sum', 'zero_loans90_ratio', 'pclose_ratio', 'fclose_ratio']

    if not os.path.exists(chunk_output):
        agg_chunk.to_parquet(chunk_output, index=False, engine='pyarrow')
    else:
        existing = pd.read_parquet(chunk_output)
        combined = pd.concat([existing, agg_chunk], ignore_index=True)
        final_combined = combined.groupby("id").agg({
            "total_loans_count": "max", "loans_limit_max": "max", "loans_outstanding_mean": "mean",
            "total_overdue_sum": "sum", "loans90_sum": "sum", "zero_loans90_ratio": "mean",
            "pclose_ratio": "mean", "fclose_ratio": "mean"
        }).reset_index()
        final_combined.to_parquet(chunk_output, index=False, engine='pyarrow')

    del chunk, agg_chunk
    gc.collect()

Всего строк в исходном файле: 7845701
Финальная агрегированная таблица успешно собрана на диске!


In [20]:
val_data = pd.read_parquet(chunk_output)

In [21]:
y_pred_proba = model.predict_proba(val_data)

In [22]:
ans = pd.DataFrame({'id': val_data['id'], 'target_value': y_pred_proba[:, 1]})

Export

In [23]:
from google.colab import files

In [24]:
ans.to_csv("scoring.csv", index=False)

In [25]:
files.download("scoring.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>